In [1]:
import requests

In [2]:
# Example coordinates: longitude, latitude

_CHITWAN_COORDINATES = {"LAT": 27.643692887895675, "LON": 84.36065413845583}
_BHAKTAPUR_COORDINATES = {"LAT": 27.597964, "LON": 85.384573}
_BHARATPUR_COORDINATES = {"LAT": 27.676546, "LON": 84.463889}

LAND_TERRAIN = {
    0 : "Terai",
    1 : "Hill",
    2 : "Mountain"
}

def get_terrain_elevation(lat: float, lon: float) -> dict:
    try:
        url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if 'elevation' in data and len(data["elevation"]) > 0:
            elevation = float(data["elevation"][0])
        else:
            raise ValueError("Elevation data could not be parsed from response.")
    
        if elevation < 1000:
            terrain = LAND_TERRAIN[0]
        elif elevation >= 1000 and elevation < 2000:
            terrain =  LAND_TERRAIN[1]
        else:
            terrain =  LAND_TERRAIN[2]
        
        return {
            "lat": lat,
            "lng": lon,
            "elevation": elevation,
            "terrain": terrain
        }

    except Exception as e:
        print(f"Warning: Elevation API request failed ({e}). Please set terrain manually.")
        return {"elevation_m": None, "terrain": "Terai"} # Safe fallback

get_terrain_elevation(_CHITWAN_COORDINATES["LAT"], _CHITWAN_COORDINATES["LON"])

{'lat': 27.643692887895675,
 'lng': 84.36065413845583,
 'elevation': 189.0,
 'terrain': 'Terai'}

In [3]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class FarmerPriority():
    """
        Farmer priority on the scale between 0-1 with normalized as .5. And only one number after decimal \n
        0.0 -> Not important \n
        0.3 -> Low importantance \n
        0.5 -> Medium importance \n
        0.8 -> High importance \n
        1.0 -> Extremely important \n

        Args:
            profit: float
            food_security: float
            soil_health: float
            water_saving: float
            low_risk: float
    """
    profit: float = 0.5
    food_security: float = 0.5
    soil_health: float = 0.5
    water_saving: float = 0.5
    low_risk: float = 0.5
    must_include: List[str] = field(default_factory=list)   # crop names
    exclude: List[str] = field(default_factory=list)
    min_food_share: float = 0.0

    _VALID_SCORE = [0.0, 0.3, 0.5, 0.8, 1]

    @staticmethod
    def _count_decimals(value: float) -> int:
        str_value = str(value)
        if "." in str_value:
            # Split by '.' and count characters on the right
            return len(str_value.split(".")[1])
        return 0

    def __post_init__(self):
        _preferences = {
            "profit": self.profit,
            "food_security": self.food_security,
            "soil_health": self.soil_health,
            "water_saving": self.water_saving,
            "low_risk": self.low_risk
        }
        for name, value in _preferences.items():
            if not (0 <= value <= 1):
                raise ValueError(
                    f"{name} must be between 0 and 1. Got {value}"
                )
            
            if value not in self._VALID_SCORE:
                raise ValueError(
                    f"{name} must be between {self._VALID_SCORE}. Got {value}"
                )
            
            if round(value, 1) != value:
                raise ValueError(
                    f"{name} must have at most 1 decimal place (e.g., 0.5, 0.6). Got {self._count_decimals(value)}"
                )

        if not (0 <= self.min_food_share <= 1 ) or (round(self.min_food_share, 2) != self.min_food_share):
            raise ValueError(
                f"min_food_share must be between 0 and 1 with at most 2 decimal place. "
                f"Got {self.min_food_share}"
            )

In [4]:
import datetime as dt
import numpy as np

@dataclass
class FieldProfile:
    """
        Defines the physical and operational characterisitics of the land.
    """
    _VALID_TERRAINS = ["Terai", "Hill", "Mountain"]
    field_id: str = "field_001"
    terrain: str = "Terai"
    irrigation: bool = False
    soil_depth_cm: float = 100.0
    start_date: dt.date = dt.date(2026, 11, 1)

    def __post_init__(self):
        if self.terrain not in self._VALID_TERRAINS:
            raise ValueError(
                    f"Invalid terrain: {self.terrain}."
                    f"Choose from {self._VALID_TERRAINS}")
        if not 0 < self.soil_depth_cm <= 200:      # the old check let 0 and negative depths through
            raise ValueError(f"soil_depth_cm must be > 0 and <= 200. Got {self.soil_depth_cm}")
        if not isinstance(self.irrigation, (bool, np.bool_)):
            raise TypeError(f"irrigation must be True/False. Got {self.irrigation!r}")
        if isinstance(self.start_date, dt.datetime) or not isinstance(self.start_date, dt.date):
            raise TypeError("start_date must be a datetime.date")

In [14]:
from dataclasses import field
from typing import Tuple, Dict

@dataclass
class RotationConfig:
    # --- candidate generation
    rotation_length: int = 3              # number of crop cycles in the rotation
    max_candidates: int = 50_000          # random-sample if the full product is larger
    random_seed: int = 27
    # --- timing
    turnaround_days: int = 10           # land prep between harvest and next sowing
    max_fallow_days: int = 120            # longest acceptable idle gap between crops
    max_horizon_days: int = 800           # whole rotation must finish within this
    max_single_crop_days: int = 300       # longer = perennial / not rotatable
    harvest_mismatch_tol_days: int = 20   # data-quality flag only
    # --- water
    rain_eff: float = 0.8                # effective-rain fraction (only if precip column exists)
    min_wet_low: float = 0.25             # required mean profile wetness at low demand (2 mm/d)
    min_wet_high: float = 0.55            # ... at high demand (6 mm/d)
    water_reject_ratio: float = 0.60      # rainfed: reject if supply/demand ratio below this
    rs_unit: str = "kWh"                  # "kWh" (kWh/m2/day) or "MJ" (MJ/m2/day)
    # --- soil
    bd_reject: float = 1.75               # bulk density (kg/dm3) at/above which roots are blocked
    clay_max_root_crops: float = 55.0     # % clay: reject tuber/root crops above this
    min_soil_depth_ratio: float = 0.5     # soil depth must be >= this fraction of rooting depth
    n_ok_g_per_kg: float = 2.0            # soil N above this = little need for N-fixers
    # --- rotation rules
    cyclic: bool = True                   # rotation repeats, so last->first also counts for family gap
    default_family_gap: int = 2           # same family must be >= this many cycles apart
    family_gap_overrides: Dict[str, int] = field(default_factory=lambda: {
        "Solanaceae": 3, "Brassicaceae": 3, "Fabaceae": 2})
    require_legume: bool = False           # at least one legume / green manure in the rotation

CFG = RotationConfig()

In [5]:
farmer = FarmerPriority(
    profit=1,
    food_security=0.8,
    soil_health=0.5,
    water_saving=0.5,
    low_risk=0.5,
    min_food_share=0
)

In [6]:
current_field_profile = FieldProfile(
    field_id="CHITWAN_COORDINATES_001",
    terrain=get_terrain_elevation(_BHARATPUR_COORDINATES["LAT"], _BHARATPUR_COORDINATES["LON"])["terrain"],
    irrigation=False,
    start_date=dt.date(2026, 11, 1)
)
current_field_profile

FieldProfile(field_id='CHITWAN_COORDINATES_001', terrain='Terai', irrigation=False, soil_depth_cm=100.0, start_date=datetime.date(2026, 11, 1))

In [7]:
def priority_weight(p: FarmerPriority):
    default_t_weight = p.profit + p.food_security + p.soil_health + p.water_saving + p.low_risk
    if default_t_weight == 0:
        raise ValueError(
                "User need to select a priority"
            )
    total_weight = max(default_t_weight, 1e-9) # max for non-zero divisible
    n = lambda x : x/total_weight # normalize the priorities
    return {
        "soil_fit": n(p.soil_health)*0.5, 
        "water_fit": n(p.water_saving),
        "timing": n(p.low_risk), 
        "rotation_health": n(p.soil_health)*0.5,
        "ndvi_support": 0.0, 
        "profit": n(p.profit), 
        "food": n(p.food_security)
    }

priority_weight(farmer)

{'soil_fit': 0.07575757575757576,
 'water_fit': 0.15151515151515152,
 'timing': 0.15151515151515152,
 'rotation_health': 0.07575757575757576,
 'ndvi_support': 0.0,
 'profit': 0.30303030303030304,
 'food': 0.24242424242424246}

In [8]:
from pathlib import Path
import pandas as pd

crop_path = Path().resolve().parent / "data" / "Crop" / "cleaned" / "combined_crop_table.csv"
crop_df = pd.read_csv(crop_path)
crop_df

,Crop Name,Botanical Species Name,Botanical Family,Primary Agronomic Function,Terai,Hill,Mountain,Rooting depth (m),Category,Optimal pH Min,Optimal pH Max,Absolute pH Min,Absolute pH Max,Agronomic Notes
0,Rice,Oryza sativa L.,Poaceae,Cash / Staple Food,Yes,Yes,Yes,0.5,Cereals & pseudocereals,5.5,7.0,4.5,9.0,"Highly adaptable to flooded, wet soils across ..."
1,Rice,Oryza sativa L.,Poaceae,Cash / Food,No,Yes,No,0.5,Cereals & pseudocereals,5.5,7.0,4.5,9.0,"Highly adaptable to flooded, wet soils across ..."
2,Wheat,Triticum aestivum L.,Poaceae,Cash / Staple Food,Yes,Yes,Yes,1.0,"Cereals & pseudocereals, forage/pasture, medic...",6.0,7.0,5.5,8.5,"Does best in neutral to slightly acidic, ferti..."
3,Corn,Zea mays L.,Poaceae,Cash / Food / Fodder,Yes,Yes,Yes,0.9,Cereals & pseudocereals,5.0,7.0,4.5,8.5,Sensitive to high acidity; performs poorly if ...
4,Barley,Hordeum vulgare L.,Poaceae,Cash / Food,No,Yes,Yes,1.0,"Cereals & pseudocereals, medicinals & aromatic",6.5,7.5,6.0,8.0,Excellent tolerance to alkaline and moderately...
5,Barley,Hordeum vulgare var. nudum,Poaceae,Food Crop,No,No,Yes,1.0,Cereals & pseudocereals,6.5,7.5,6.0,8.0,Excellent tolerance to alkaline and moderately...
6,Finger Millet,Eleusine coracana (L.) Gaertn.,Poaceae,Food / Cash Crop,No,Yes,Yes,1.0,"Cereals & pseudocereals, forage/pasture, envir...",6.0,7.0,5.5,8.2,"Hardy crop that grows well in medium-textured,..."
7,Oat,Avena sativa L.,Poaceae,Fodder / Cash Crop,No,Yes,No,1.0,"Cereals & pseudocereals, cover crop",5.0,6.0,4.5,7.5,Moderately tolerant of acidic conditions compa...
8,Soybean,Glycine max (L.) Merr.,Fabaceae,Cash / Food Crop,No,Yes,No,0.6,Pulses (grain legumes),5.5,6.5,4.5,8.4,Prefers slightly acidic soils; crucial for eff...
9,Lentil,Lens culinaris Medik.,Fabaceae,Cash Pulse Crop,Yes,No,No,0.6,"Pulses (grain legumes), medicinals & aromatic",5.5,7.5,4.5,8.2,Thrives in neutral soils close to pH 7; sensit...


In [9]:
def terrain_check(field: FieldProfile, crop):

    if not field or not getattr(field, 'terrain', None):
        print("Error: Field object or terrain attribute is missing.")
        return False
    
    terrain_col = str(field.terrain).strip().lower()

    if terrain_col not in crop.index:
        print(f"Terrain column '{terrain_col}' not found in crop data.")
        return False
    
    suitability = crop[terrain_col]
    truthy_values = {'yes', 'true', 'y', '1', True, 1}

    if isinstance(suitability, str):
        suitability_check = suitability.strip().lower()
    else:
        suitability_check = suitability

    if suitability_check in truthy_values:
        return True
    else:
        # Safely extract crop name (falls back to index name or first column)
        # crop_name = (
        #     crop.crop 
        #     if hasattr(crop, 'crop') and crop.crop is not None 
        #     else crop.iloc[0]
        # )
        # print(f"{crop_name} not recommended for '{field.terrain}' terrain")
        return False

In [10]:
crop_df.columns

Index(['Crop Name', 'Botanical Species Name', 'Botanical Family',
       'Primary Agronomic Function', 'Terai', 'Hill', 'Mountain',
       'Rooting depth (m)', 'Category', 'Optimal pH Min', 'Optimal pH Max',
       'Absolute pH Min', 'Absolute pH Max', 'Agronomic Notes'],
      dtype='str')

In [11]:
ATTR_RENAME = {
    "Crop Name": "crop", 
    "Botanical Species Name": "species", 
    "Botanical Family": "family",
    "Primary Agronomic Function": "function", 
    "Terai": "terai", 
    "Hill": "hill", 
    "Mountain": "mountain",
    "Rooting depth (m)": "rooting_depth_m", 
    "Category": "category",
    "Optimal pH Min": "ph_opt_min", 
    "Optimal pH Max": "ph_opt_max",
    "Absolute pH Min": "ph_abs_min", 
    "Absolute pH Max": "ph_abs_max", 
    "Agronomic Notes": "agronomic_notes",
}

crop_df = crop_df.rename(columns=ATTR_RENAME)
crop_df.columns

Index(['crop', 'species', 'family', 'function', 'terai', 'hill', 'mountain',
       'rooting_depth_m', 'category', 'ph_opt_min', 'ph_opt_max', 'ph_abs_min',
       'ph_abs_max', 'agronomic_notes'],
      dtype='str')

In [12]:
import pandas as pd
import glob
import os

COMPLETED_TASK_ID="8d1573c7-1f8b-449f-848c-dcf754f92547"

def get_modis():

    output_dir = os.path.join("..", "data", "MODIS", COMPLETED_TASK_ID)

    CSV_PATH = os.path.join(output_dir, "*results*.csv")

    # 1. Automatically find the CSV file in your current folder
    csv_files = glob.glob(CSV_PATH)
    if not csv_files:
        print("No CSV file found! Make sure you are running this in the folder where files were downloaded.")
        raise FileNotFoundError(
            f"No MODIS results CSV found in {output_dir}"
        )
        
    csv_filename = csv_files[0]
    print(f"Loading data from: {csv_filename}")

    # 2. Read the CSV file into a Pandas DataFrame
    df = pd.read_csv(csv_filename)

    # Display available columns to understand the structure
    print("\n--- Available Columns in CSV ---")
    print(df.columns.tolist())

    # 3. Process and Clean the Data

    # 1. Convert Date column to actual datetime objects
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date') # Sort chronologically for time-series analysis

    # 2. Define exact column names from your dataset
    MODIS_NDVI_COL = 'MOD13Q1_061__250m_16_days_NDVI'
    MODIS_EVI_COL = 'MOD13Q1_061__250m_16_days_EVI'
    MODIS_QA_COL = 'MOD13Q1_061__250m_16_days_pixel_reliability'

    # fail early with a clear message (previously a missing column surfaced as NameError/KeyError later)
    missing = [c for c in (MODIS_NDVI_COL, MODIS_EVI_COL, MODIS_QA_COL, "Date") if c not in df.columns]
    if missing:
        raise KeyError(f"MODIS CSV is missing required columns: {missing}")

    # 3. Point value scale
    if MODIS_NDVI_COL in df.columns and MODIS_EVI_COL in df.columns:
        df['Clean_NDVI'] = df[MODIS_NDVI_COL]
        df['Clean_EVI'] = df[MODIS_EVI_COL]

    # 4. Evaluate Pixel Quality and Missing Data/Cloud Coverage
    if MODIS_QA_COL in df.columns:
        print("\nChecking pixel reliability and data coverage...")
        
        # MODIS Pixel Reliability mapping:
        # 0 = Good data, 1 = Marginal, 2 = Snow/ice, 3 = Cloudy, -1 = No data / Fill value
        print(df[MODIS_QA_COL].value_counts())
        
        # Filter for good or marginal data only (optional)
        valid_mask = df[MODIS_QA_COL].isin([0, 1]) # list(cfg.ok_reliability)
        filtered_df = df[valid_mask].copy()
        
        print(f"\nTotal observations: {len(df)}")
        print(f"Valid/Good observations after QA filter: {len(filtered_df)}")

    # Valid NDVI and EVI observations
    ndvi_valid_share = float(
        (valid_mask & df["Clean_NDVI"].notna()).mean()
    )

    evi_valid_share = float(
        (valid_mask & df["Clean_EVI"].notna()).mean()
    )

    return filtered_df, ndvi_valid_share, evi_valid_share

In [15]:
import numpy as np

CAL = pd.date_range("2000-01-01", "2070-12-31", freq="D")
ORD0 = dt.date(2000, 1, 1).toordinal()
DEFAULT_AWC_FRAC = 0.12           # fallback plant-available water fraction (loam-ish) when wv33/wv1500 missing
def doy_365(dates):
    """
    Convert dates to a consistent 1..365 day-of-year.
    February 29 is grouped with February 28.
    """
    dates = pd.to_datetime(dates)
    doy = dates.dt.dayofyear.copy()
    # Remove leap-year shift after February
    after_feb = dates.dt.is_leap_year & (dates.dt.month > 2)
    doy = doy - after_feb.astype(int)
    # Map February 29 to February 28
    feb29 = (dates.dt.month == 2) & (dates.dt.day == 29)
    doy.loc[feb29] = 59

    return doy

DOY_IDX = doy_365(pd.Series(CAL)).to_numpy() - 1

def _circ_smooth(a, w=15):
    a = np.asarray(a, dtype=float)
    if len(a) != 365:
        raise ValueError("Circular smoothing expects 365 values.")
    p = w // 2
    ext = np.r_[a[-p:], a, a[:p]]
    return np.convolve(
        ext,
        np.ones(w) / w,
        mode="valid"
    )

class Site:
    """
        Soil profile + 365-day climatology + cumulative arrays for O(1) season sums.
    """
    def __init__(self, power_df: pd.DataFrame, soilgrid_df: pd.DataFrame, cfg: RotationConfig):
        self._power_df = power_df.copy()
        self._soil_df = soilgrid_df.copy()
        
        # --- MODIS quality filter (NDVI/EVI only)
        self._modis_df, self.ndvi_valid_share, self.evi_valid_share = get_modis()

        # --- static soil from SoilGirds
        soil = self._prepare_soil(self._soil_df)
        self._soil = soil

        # Calculate plant-available water capacity
        wv33 = self._soil.get("wv33", np.nan)
        wv15 = self._soil.get("wv1500", np.nan)

        if pd.notna(wv33) and pd.notna(wv15):
            divisor = 1000 if wv33 > 100 else (100 if wv33 > 1 else 1)   # raw 10^-3 cm3/cm3, %, or fraction
            self.awc_frac = max(
                (wv33 - wv15) / divisor,
                0.0
            )
        else:
            self.awc_frac = DEFAULT_AWC_FRAC

        # Prepare NASA POWER weather

        self._power_df["date"] = pd.to_datetime(self._power_df["date"],  errors="coerce")
        self._power_df = (
            self._power_df
            .dropna(subset=["date"])
            .sort_values("date")
            .reset_index(drop=True)
        )
        self.n_weather_days = len(self._power_df)
        # Calculate consistent 365-day DOY
        self._power_df["doy"] = doy_365(self._power_df["date"]).values
        # Calculate mean temperature if missing
        if ("tmean" not in self._power_df or self._power_df["tmean"].isna().all()):
            self._power_df["tmean"] = (self._power_df["tmax"] + self._power_df["tmin"])/2
        # Daily weather variables
        weather_vars = [
            v for v in [
                "tmean", "tmax", "tmin",
                "rh", "wind", "rs", "wet", "precip"
            ]
            if v in self._power_df.columns
        ]
        if not {"tmean", "rs"}.issubset(weather_vars):
            raise ValueError(
                "NASA POWER data must contain tmean and rs."
            )
        
        weather_clim = (
            self._power_df
            .groupby("doy")[weather_vars]
            .mean()
            .reindex(range(1, 366))
            .interpolate(limit_direction="both")
        )
        # Reject missing weather climatology
        if weather_clim.isna().any().any():
            raise ValueError(
                "Weather climatology still contains missing values."
            )
        for v in weather_clim.columns:
            weather_clim[v] = _circ_smooth(weather_clim[v].values)
        
        # Calculate reference evapotranspiration
        rs_mj = (weather_clim["rs"] * 3.6 if cfg.rs_unit == "kWh" else weather_clim["rs"])
        weather_clim["et0"] = np.clip(0.0135 * (weather_clim["tmean"] + 17.8) * rs_mj / 2.45, 0, None)
        self.weather_clim = weather_clim
        self.has_precip = "precip" in weather_clim.columns
        # Map 365-day climatology onto the full calendar
        self.daily = {
            variable_name: weather_clim[variable_name].to_numpy()[DOY_IDX]
            for variable_name in weather_clim.columns
        }

        # Cumulative sums for fast seasonal calculations
        self.cum = {
            variable_name: np.r_[0.0, np.cumsum(daily_values)]
            for variable_name, daily_values in self.daily.items()
        }
        self.prepare_modis_daily()
    
    @staticmethod
    def _prepare_soil(df):
        d = df.copy()
        aliases = {
            "phh2o": "ph",
            "wv0010": "wv10",
            "wv0033": "wv33",
            "bdod": "bdod"
        }

        # Long format: Property, Depth, Value, Unit
        if {"Property", "Depth", "Value"}.issubset(d.columns):
            d["Property"] = (d["Property"].astype(str).str.lower())

            # Current prototype uses the 30-60 cm layer.
            # This is not a complete root-zone profile.
            surface = d[d["Depth"].astype(str) == "30-60cm"].copy()

            if surface.empty:
                raise ValueError(
                    "No 30-60cm SoilGrids layer found. "
                    "Choose and aggregate soil depths explicitly."
                )
            s = (
                surface
                .groupby("Property")["Value"]
                .median()
            )
            s = s.rename(index=aliases)

        # Wide format
        else:
            rename = {
                k: v for k, v in aliases.items()
                if k in d.columns
            }

            d = d.rename(columns=rename)

            s = d.select_dtypes(
                include=np.number
            ).median()

        # Convert SoilGrids pH*10 if necessary
        if "ph" in s and pd.notna(s["ph"]):
            if s["ph"] > 14:
                s["ph"] = s["ph"] / 10
        return s
    
    def prepare_modis_daily(self):
        self.modis_dataframe = self._modis_df.copy()
        modis_dataframe = self.modis_dataframe.rename(columns={
            "Date": "date",
            "Clean_NDVI": "ndvi",
            "Clean_EVI": "evi"
        })
        modis_dataframe["date"] = pd.to_datetime(
            modis_dataframe["date"],
            errors="coerce"
        )
        for variable_name in ["ndvi", "evi"]:
            modis_dataframe[variable_name] = pd.to_numeric(
                modis_dataframe[variable_name],
                errors="coerce"
            )
        # Remove rows without valid dates
        modis_dataframe = modis_dataframe.dropna(
            subset=["date"]
        ).copy()

        # Convert all observation dates to a consistent 365-day calendar
        modis_dataframe["doy"] = doy_365(
            modis_dataframe["date"]
        ).to_numpy()

        # Store historical seasonal climatologies
        self.modis_climatology = {}

        for variable_name in ["ndvi", "evi"]:
            observations = (
                modis_dataframe
                .dropna(subset=[variable_name])
                .groupby("doy")[variable_name]
                .mean()
            )
            # Reindex to all 365 days
            seasonal_climatology = observations.reindex(
                range(1, 366)
            )
            # Find days with actual historical observations
            valid_days = seasonal_climatology.notna().to_numpy()
            if valid_days.sum() < 2:
                raise ValueError(
                    f"Not enough historical {variable_name.upper()} "
                    "observations to build seasonal climatology."
                )
            # Circular interpolation:
            # allows missing values near December/January
            # to be interpolated using observations across the year boundary.
            known_positions = np.where(valid_days)[0]
            known_values = seasonal_climatology.to_numpy()[valid_days]

            extended_positions = np.concatenate([
                known_positions - 365,
                known_positions,
                known_positions + 365
            ])
            extended_values = np.concatenate([
            known_values,
            known_values,
            known_values
            ])

            all_positions = np.arange(365)

            interpolated_values = np.interp(
                all_positions,
                extended_positions,
                extended_values
            )
            # Smooth the seasonal curve
            smoothed_values = _circ_smooth(
                interpolated_values,
                w=15
            )

            # Store the 365-day climatology
            seasonal_climatology = pd.Series(
                smoothed_values,
                index=range(1, 366)
            )
            self.modis_climatology[variable_name] = (
            seasonal_climatology
            )

            # Map the historical seasonal pattern to every calendar date,
            # including dates after the final satellite observation.
            future_daily_values = seasonal_climatology.to_numpy()[DOY_IDX]

            self.daily[variable_name] = future_daily_values

            # Build cumulative sums for fast seasonal calculations
            self.cum[variable_name] = np.r_[
                0.0,
                np.cumsum(future_daily_values)
            ]

            # Every calendar day has a climatology value now.
            # This count represents climatology availability,
            # NOT actual daily satellite observations.
            self.cum[f"{variable_name}_count"] = np.r_[
                0,
                np.cumsum(
                    np.isfinite(future_daily_values).astype(int)
                )
            ]
        # Calculate maximum NDVI once, outside the variable loop
        self.ndvi_max = float(
            np.max(self.daily["ndvi"])
        )


    def span_sum(self, variable_name: str, start_date: dt.date, end_date: dt.date):
        """
            Calculate the total value of a daily variable between two dates.
            The interval is start-inclusive and end-exclusive: [start_date, end_date).
        """
        start_day_index = start_date.toordinal() - ORD0
        end_day_index = end_date.toordinal() - ORD0
        cumulative_end_value = self.cum[variable_name][end_day_index]
        cumulative_start_value = self.cum[variable_name][start_day_index]
        total_value = cumulative_end_value - cumulative_start_value

        return float(total_value)
    
    def span_mean(self, variable_name: str, start_date: dt.date, end_date: dt.date):
        """
            Calculate the average daily value of a variable between two dates.
            The interval is start-inclusive and end-exclusive: [start_date, end_date).
        """
        start_index = start_date.toordinal() - ORD0
        end_index = end_date.toordinal() - ORD0

        total_value = (
            self.cum[variable_name][end_index]
            - self.cum[variable_name][start_index]
        )

        if variable_name in ["ndvi", "evi"]:
            valid_day_count = (
                self.cum[f"{variable_name}_count"][end_index]
                - self.cum[f"{variable_name}_count"][start_index]
            )

            if valid_day_count == 0:
                return np.nan

            return total_value / valid_day_count

        number_of_days = (end_date - start_date).days

        if number_of_days <= 0:
            raise ValueError("End date must be after start date.")

        return total_value / number_of_days


/var/folders/bl/c9xq73wj61b5k_gjtrh9b0zh0000gn/T/ipykernel_16253/3420647529.py:12: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  doy = dates.dt.dayofyear.copy()


In [17]:
NASA_POWER_DF = pd.read_csv(Path().resolve().parent / "data" / "Crop" / "cleaned" / "nasa_power.csv.csv")
SOIL_GRID_DF = pd.read_csv(Path().resolve().parent / "data" / "Crop" / "cleaned" / "soil_grid.csv.csv")
site = Site(NASA_POWER_DF, SOIL_GRID_DF, CFG)

Loading data from: ../data/MODIS/8d1573c7-1f8b-449f-848c-dcf754f92547/Chitwan-MOD13Q1-2021-01-01-to-2025-12-12-MOD13Q1-061-results.csv

--- Available Columns in CSV ---
['ID', 'Latitude', 'Longitude', 'Date', 'MODIS_Tile', 'MOD13Q1_061_Line_Y_250m', 'MOD13Q1_061_Sample_X_250m', 'MOD13Q1_061__250m_16_days_EVI', 'MOD13Q1_061__250m_16_days_NDVI', 'MOD13Q1_061__250m_16_days_VI_Quality', 'MOD13Q1_061__250m_16_days_pixel_reliability', 'MOD13Q1_061__250m_16_days_VI_Quality_bitmask', 'MOD13Q1_061__250m_16_days_VI_Quality_MODLAND', 'MOD13Q1_061__250m_16_days_VI_Quality_MODLAND_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_VI_Usefulness', 'MOD13Q1_061__250m_16_days_VI_Quality_VI_Usefulness_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Aerosol_Quantity', 'MOD13Q1_061__250m_16_days_VI_Quality_Aerosol_Quantity_Description', 'MOD13Q1_061__250m_16_days_VI_Quality_Adjacent_cloud_detected', 'MOD13Q1_061__250m_16_days_VI_Quality_Adjacent_cloud_detected_Description', 'MOD13Q1_061__250m_16_day

/var/folders/bl/c9xq73wj61b5k_gjtrh9b0zh0000gn/T/ipykernel_16253/3420647529.py:12: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  doy = dates.dt.dayofyear.copy()
/var/folders/bl/c9xq73wj61b5k_gjtrh9b0zh0000gn/T/ipykernel_16253/3420647529.py:12: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  doy = dates.dt.dayofyear.copy()


In [18]:
# ---- soil (date independent, cached per crop) --------------------------------
def soil_check(crop, site: Site, field: FieldProfile, cfg: RotationConfig):
    """
        Evaluates whether a particular crop is suitable for the field's soil conditions.
        Checks:
        1.  Soil pH (acidity/alkalinity).
        2.  Soil depth (whether roots have enough space).
        3.  Soil texture (clay and sand percentages).
        4.  Soil compaction (bulk density).
        then calculates a soil suitability score between 0 and 1.
    """
    s, reasons = site._soil, []
    ph = float(s.ph)
    # check whether the given crop ph falls under soil ph.
    if ph < crop.ph_abs_min or ph > crop.ph_abs_max:
        reasons.append(f"soil pH {ph:.1f} outside {crop.crop}'s absolute range {crop.ph_abs_min}-{crop.ph_abs_max}")

    if crop.ph_opt_min <= ph <= crop.ph_opt_max: 
        ph_score = 1.0 # optimal range
    elif ph < crop.ph_opt_min:
        ph_score = (ph - crop.ph_abs_min)/max(crop.ph_opt_min - crop.ph_abs_min, 1e-6) # how close ph is to the optimal minimum
    else: 
        ph_score = (crop.ph_abs_max - ph)/max(crop.ph_abs_max - crop.ph_opt_max, 1e-6) # how close ph is to the optimal maximum

    ph_score = float(np.clip(ph_score, 0, 1)) # keeps the ph between 0-1 range.

    # check whether the soil is deep enough for the crop's roots.
    if field.soil_depth_cm < cfg.min_soil_depth_ratio * crop.rooting_depth_m * 100:
        reasons.append(f"soil depth {field.soil_depth_cm:.0f} cm too shallow for rooting depth {crop.rooting_depth_m * 100:.0f} cm")
    depth_score = float(np.clip(field.soil_depth_cm / (crop.rooting_depth_m * 100), 0, 1))

    # lay, sand and slit percentages.
    clay, sand, silt = float(s.clay), float(s.sand), float(s.silt)

    # Check whether soil fractions are valid
    if not all(0 <= x <= 100 for x in [clay, sand, silt]):
        raise ValueError("Soil texture percentages must be between 0 and 100.")
    
    # Soil fractions should approximately sum to 100%
    texture_total = clay + sand + silt

    if not 95 <= texture_total <= 105:
        raise ValueError(
            f"Invalid soil texture composition: "
            f"clay + sand + silt = {texture_total:.1f}%"
        )

    # check whether the crop belongs to a tuber or root category.
    root_crop = any(k in crop.category.lower() for k in ["tuber", "root"])
    if root_crop and clay > cfg.clay_max_root_crops:
        reasons.append(f"clay {clay:.0f}% too heavy for tuber/root crop")
    
    # Texture penalty score
    tex_penalty = 0.0

    if clay > 50:
        tex_penalty += 0.4

    if sand > 70:
        tex_penalty += 0.3

    if silt > 80:
        tex_penalty += 0.2

    tex_score = float(np.clip(1.0 - tex_penalty, 0, 1))

    # check soil compaction
    bd = float(s.bdod)
    if bd >= cfg.bd_reject:
        reasons.append(f"bulk density {bd:.2f} kg/dm3 restricts root growth")

    # Calculate bulk density score
    bd_score = 0.7 if bd > 1.55 else 1.0

    # Calculate final soil fit
    fit = float(np.mean([ph_score, depth_score, tex_score, bd_score]))

    return (
        not reasons,
        "; ".join(reasons),
        dict(soil_fit=fit, ph_score=ph_score)
    )

In [19]:
import itertools

def generate_candidates(number_of_crops: int, rotation_length: int, max_candidates: int, seed: int):
    """
        Generates possible crop rotations.

        Args:
            number_of_crops(int): number of available crops
            rotation_length(int): number of crops in each rotation
            max_candidates: maximum number of candidates to generates
            seed: random seed
    """
    if number_of_crops <= 0 or rotation_length <= 0 or max_candidates <= 0:
        raise ValueError("number_of_crops, rotation_length, and max_candidates must be positive.")

    total = number_of_crops ** rotation_length # total possible rotations

    if total <= max_candidates:
        return list(itertools.product(range(number_of_crops), repeat=rotation_length)) # generates every possible sequence of crop indices.
    
    random_generator = np.random.default_rng(seed)

    unique_candidates = set()
    # check how many unique candidates currently exist
    while len(unique_candidates) < max_candidates:
        # check how many we needs and get X2 more random sequences containing sequences until 10,000 in each batch
        remaining_candidates = max_candidates - len(unique_candidates)
        batch_size = min(remaining_candidates * 2, 10000)

        samples = random_generator.integers(
            0,
            number_of_crops,
            size=(batch_size, rotation_length)
        )

        unique_candidates.update(map(tuple, samples))

    return sorted(unique_candidates)[:max_candidates] 

In [20]:
import calendar

def mk_date(y, m, d):
    """
        Helper function to return datetime format.
    """
    month = int(m)
    day = int(d)
    last_day = calendar.monthrange(year=y, month=month)[1]

    return dt.date(
        y,
        month,
        min(day, last_day)
    )

# ---- planting window ------------------------------------------------------

def candidate_sowing_dates(
    field_ready_date: dt.date,
    crop,
    crop_calender_df,
    terrain_str,
    max_fallow_days=120,
    step_days=7
):
    """
    Returns multiple valid sowing dates for a crop.

    Searches all calendar windows within the allowed
    fallow period, rather than selecting only the earliest.
    """
    crop_name = str(
        crop.crop if hasattr(crop, "crop") else crop
    ).strip().lower()

    terrain = str(terrain_str).strip().lower()

    filtered = crop_calender_df[
        (crop_calender_df["crop"].astype(str).str.strip().str.lower() == crop_name) &
        (crop_calender_df["aez"].astype(str).str.strip().str.lower() == terrain)
    ]

    if filtered.empty:
        print(f"[NO CALENDAR MATCH] {crop_name}, {terrain}")
        return []

    latest_allowed = field_ready_date + dt.timedelta(
        days=max_fallow_days
    )

    candidates = set()

    for idx, row in filtered.iterrows():

        try:
            # Year-round crops
            all_year = str(row["all_year"]).strip().lower()

            if all_year in ("yes", "true", "1", "y"):
                current = field_ready_date

                while current <= latest_allowed:
                    candidates.add((current, idx))
                    current += dt.timedelta(days=step_days)

                continue

            early_month = int(row["sow_early_month"])
            early_day = int(row["sow_early_day"])
            late_month = int(row["sow_late_month"])
            late_day = int(row["sow_late_day"])

            # Check all relevant calendar years
            for year in range(
                field_ready_date.year - 1,
                latest_allowed.year + 2
            ):

                window_start = mk_date(
                    year, early_month, early_day
                )

                window_end = mk_date(
                    year, late_month, late_day
                )

                # Handle windows crossing into the next year
                if window_end < window_start:
                    window_end = mk_date(
                        year + 1, late_month, late_day
                    )

                # Intersect calendar window with allowed dates
                actual_start = max(
                    window_start, field_ready_date
                )

                actual_end = min(
                    window_end, latest_allowed
                )

                if actual_start > actual_end:
                    continue

                current = actual_start

                while current <= actual_end:
                    candidates.add((current, idx))
                    current += dt.timedelta(days=step_days)

                # Include the final valid date if the interval
                # does not end exactly on a seven-day step.
                if actual_end >= actual_start:
                    candidates.add((actual_end, idx))

        except (ValueError, TypeError, KeyError) as e:
            print(
                f"[INVALID CALENDAR ROW] index={idx}, "
                f"crop={crop_name}, error={e}"
            )

    result = sorted(
        candidates,
        key=lambda x: (x[0], x[1])
    )
    return result

In [21]:
# ---- water (depends on sowing date, cached per crop+date) -----------------------
def water_check(crop, sowing_date: dt.date, harvest_date: dt.date, site: Site, field: FieldProfile, cfg: RotationConfig):
    # Validate the growing period
    if harvest_date <= sowing_date:
        raise ValueError(
            f"Harvest date must be after sowing date for {crop.crop}."
        )
    growing_days = (harvest_date - sowing_date).days

    # Calculate estimated crop water demand
    reference_et0_mm = site.span_sum(
        "et0", sowing_date, harvest_date
    )
    crop_water_demand_mm = reference_et0_mm * crop.kc_mean

    # Determine effective crop rooting depth
    effective_root_depth_cm = min(crop.rooting_depth_m * 100, field.soil_depth_cm)

    # Available water storage in the root zone
    available_water_storage_mm = (site.awc_frac * effective_root_depth_cm * 10)
    initial_soil_wetness = site.span_mean("wet",sowing_date,sowing_date + dt.timedelta(days=10))

    # Check if the water is rainfed or not
    if site.has_precip:
        # when the site has rainfall
        total_rainfall_mm = site.span_sum("precip", sowing_date, harvest_date)
        effective_rainfall_mm = (cfg.rain_eff * total_rainfall_mm)
        estimated_water_supply_mm = (available_water_storage_mm * initial_soil_wetness+ effective_rainfall_mm)
        water_adequacy_ratio = (estimated_water_supply_mm/ max(crop_water_demand_mm, 1e-6))
        water_assessment_method = "water_balance"
    else:
        # when the rainfall is unavailable
        average_daily_water_demand_mm = (crop_water_demand_mm / growing_days)
        required_soil_wetness = np.interp(average_daily_water_demand_mm,[2, 6],[cfg.min_wet_low, cfg.min_wet_high])
        average_growing_season_wetness = site.span_mean("wet", sowing_date, harvest_date)
        water_adequacy_ratio = (average_growing_season_wetness/ max(required_soil_wetness, 1e-6))
        water_assessment_method = "wetness_proxy"

    # check whether the crop passed the water check
    passes_water_check = (field.irrigation or water_adequacy_ratio >= cfg.water_reject_ratio)
    if passes_water_check:
        rejection_reason = ""
    elif water_assessment_method == "water_balance":
        rejection_reason = (
            f"{crop.crop} sown {sowing_date:%d %b}: "
            f"estimated water supply/demand ratio "
            f"{water_adequacy_ratio:.2f} "
            f"is below {cfg.water_reject_ratio:.2f} "
            f"(estimated demand: {crop_water_demand_mm:.0f} mm, "
            f"rainfed field)."
        )
    else:
        rejection_reason = (
            f"{crop.crop} sown {sowing_date:%d %b}: "
            f"soil-wetness adequacy ratio "
            f"{water_adequacy_ratio:.2f} "
            f"is below {cfg.water_reject_ratio:.2f} "
            f"(rainfall data unavailable; wetness proxy used)."
        )
    water_metrics = {
        "water_ratio": float(water_adequacy_ratio),
        "demand_mm": float(crop_water_demand_mm),
        "water_mode": water_assessment_method
    }
    return passes_water_check, rejection_reason, water_metrics

In [22]:
def rotation_check(rotation_crops, cfg: RotationConfig, soil: pd.Series):
    """
        Checks the health of a crop rotation.

        Args:
            rotation_crops: Crops in the proposed rotation
            cfg: Rotation settings
            soil: Soil-property series
    """
    rotation_length = len(rotation_crops)

    if rotation_length == 0:
        raise ValueError("Rotation must contain at least one crop.")

    rejection_reasons = []

    # compare every pair of crops
    for first_index in range(rotation_length): # first crop
        for second_index in range(first_index + 1, rotation_length): # every crop after
            first_crop = rotation_crops[first_index]
            second_crop = rotation_crops[second_index]
            
            cycle_distance = second_index - first_index # check how many rotation position apart

            # check whether the rotation are cyclic or not and get minimum distance between each crop
            if cfg.cyclic:
                cycle_distance = min(cycle_distance, rotation_length - cycle_distance)

            # check whether same specifies crop are recommended close together
            if first_crop.species == second_crop.species and cycle_distance < 2:
                rejection_reasons.append(f"same species ({first_crop.species}) repeated {cycle_distance} cycle(s) apart")

            # check whether same family crop are recommended close together   
            elif first_crop.family == second_crop.family:
                required_family_gap = cfg.family_gap_overrides.get(first_crop.family, cfg.default_family_gap)
                if cycle_distance < required_family_gap:
                    rejection_reasons.append(
                        f"family {first_crop.family} ({first_crop.crop} & {second_crop.crop}) only {cycle_distance} cycle(s) apart; needs >= {required_family_gap}"
                    )

    contains_legume = any(crop.is_legume for crop in rotation_crops)

    # check whether the rotation requires legume or not                
    if cfg.require_legume and not contains_legume:
        rejection_reasons.append("No legume or green manure in the rotation.")

    # Calculate the rotation score

    unique_families = {crop.family for crop in rotation_crops}

    # check the distinct family score
    family_diversity_score = (len(unique_families) / rotation_length)

    # check nitrogen score for the soil against the crop needs
    soil_nitrogen = soil.get("nitrogen", np.nan)
    if pd.isna(soil_nitrogen):
        raise ValueError(
            "Soil nitrogen is missing; cannot calculate "
            "the nitrogen-related rotation score."
        )
    nitrogen_need_score = float(
        np.clip(1 - float(soil_nitrogen) / cfg.n_ok_g_per_kg, 0, 1)
    )

    # if rotation contains no legume give 0.0 if not provide a arbitrary value
    legume_score = (0.6 + 0.4 * nitrogen_need_score if contains_legume else 0.0)

    # root depth of the crops with the soil
    rooting_depths_cm = [crop.rooting_depth_m * 100 for crop in rotation_crops]
    if rotation_length > 1 and max(rooting_depths_cm) > 0:
        consecutive_root_depth_differences = [
            abs(rooting_depths_cm[index]- rooting_depths_cm[index - 1])
            for index in range(1, rotation_length)
        ]
        average_root_depth_difference = np.mean(consecutive_root_depth_differences)
        root_depth_alternation_score = (average_root_depth_difference/ max(rooting_depths_cm))
    else:
        root_depth_alternation_score = 0.0

    # check if legume comes after a heavy feeding crop
    if rotation_length > 1:
        legume_after_heavy_feeder_score = np.mean([
            1.0 if (rotation_crops[index].is_legume and rotation_crops[index - 1].heavy_feeder) else 0.0
            for index in range(1, rotation_length)
        ])
    else:
        legume_after_heavy_feeder_score = 0.0
    
    # combine health component
    rotation_health_score = (
        0.35 * family_diversity_score
        + 0.35 * legume_score
        + 0.15 * min(
            root_depth_alternation_score * 2, 1
        )
        + 0.15 * legume_after_heavy_feeder_score
    )

    return rejection_reasons, float(rotation_health_score)

In [23]:
import re

def parse_gp_value(gp_val):
    """
        Parses gp_value which could be an int, float, or a string range like '100 - 120' or '110-130'.
        Returns the maximum integer found, or 0 if it cannot be parsed.
    """
    if pd.isna(gp_val):
        return 0
    if isinstance(gp_val, (int, float)):
        return int(gp_val)
    
    # Convert to string and search for numbers
    val_str = str(gp_val)
    numbers = re.findall(r"\d+(?:\.\d+)?", val_str) 
    if not numbers:
        return 0
    
    # If it's a range like "100-120", take the upper bound (or lower, depending on how strict you want to be)
    int_numbers = [int(n) for n in numbers]
    return max(int_numbers)

In [24]:
def get_growing_period(crop, calender_df, field):
    filtered_rows = calender_df[
            (calender_df["crop"].astype(str).str.strip().str.lower() == str(crop.crop).strip().lower()) &
            (calender_df["aez"].astype(str).str.strip().str.lower() == str(field.terrain).strip().lower())
        ]
    if filtered_rows.empty:
        raise ValueError(
            f"No growing period for given crop:{crop.crop}"
        )     
    return parse_gp_value(filtered_rows.iloc[0]["gp_value"])

In [25]:
def assess_rotation(
    crops_df: pd.DataFrame,
    crop_calender_df: pd.DataFrame,
    site: Site,
    field: FieldProfile,
    prio: FarmerPriority,
    cfg: RotationConfig,
    top_n: int = 10
):
    """
    Assess crop rotations using multi-date sowing search and beam search.

    Each crop sequence can produce multiple possible schedules.
    Instead of accepting only the earliest sowing date, the function
    explores several valid sowing dates and retains the best partial
    schedules at each rotation stage.
    """

    WEIGHTS = priority_weight(prio)

    reject_crop = []
    accepted_crop = []

    # ---------------------------------------------------------
    # 1. PRE-SCREEN CROPS
    # ---------------------------------------------------------

    for i, crop in crops_df.iterrows():

        meta_data = []

        # Farmer exclusion
        if crop.crop in prio.exclude:
            meta_data.append(
                ("farmer_priority", "excluded by farmer")
            )

        # Terrain suitability
        ok = terrain_check(field, crop)

        if not ok:
            reason = f"{crop.crop} not recommended for {field.terrain} terrain"

            reject_crop.append({
                "crop": crop.crop,
                "checks": "terrain",
                "reasons": reason
            })
            continue

        # Calendar availability
        filtered_rows = crop_calender_df[
            (crop_calender_df["crop"].astype(str).str.strip().str.lower()
             == str(crop.crop).strip().lower()) &
            (crop_calender_df["aez"].astype(str).str.strip().str.lower()
             == str(field.terrain).strip().lower())
        ]

        if filtered_rows.empty:
            reject_crop.append({
                "crop": crop.crop,
                "checks": "season_duration",
                "reasons": f"No calendar data found for {crop.crop} in {field.terrain}"
            })
            continue

        # Crop duration
        parsed_gps = filtered_rows["gp_value"].apply(parse_gp_value)
        max_gp = parsed_gps.max()

        if pd.isna(max_gp) or max_gp > cfg.max_single_crop_days:
            reject_crop.append({
                "crop": crop.crop,
                "checks": "season_duration",
                "reasons": (
                    f"{crop.crop} needs ~{max_gp} days "
                    f"(maximum allowed: {cfg.max_single_crop_days}); "
                    "perennial/long-duration or invalid duration"
                )
            })
            continue

        # Soil suitability
        ok, reason, _ = soil_check(crop, site, field, cfg)

        if not ok:
            meta_data.append(("soil", reason))

        if meta_data:
            reject_crop.append({
                "crop": crop.crop,
                "checks": ", ".join(x[0] for x in meta_data),
                "reasons": "; ".join(x[1] for x in meta_data)
            })
        else:
            accepted_crop.append(i)

    if not accepted_crop:
        raise ValueError("No crop accepted for the rotation")

    # ---------------------------------------------------------
    # 2. PREPARE CROP POOL AND CACHES
    # ---------------------------------------------------------

    crop_pool = crops_df.loc[accepted_crop].reset_index(drop=True)

    pool_rows = [
        crop_pool.loc[i]
        for i in crop_pool.index
    ]

    soil_cache = {
        i: soil_check(crop_pool.loc[i], site, field, cfg)[2]
        for i in crop_pool.index
    }

    water_cache = {}

    rot_accpt = []
    rot_reject = []

    # Search settings
    beam_width = max(1, int(getattr(cfg, "beam_width", 100)))
    date_step_days = 7

    explored_schedules = 0

    # ---------------------------------------------------------
    # 3. GENERATE CROP SEQUENCES
    # ---------------------------------------------------------

    for seq in generate_candidates(
        len(crop_pool),
        cfg.rotation_length,
        cfg.max_candidates,
        cfg.random_seed
    ):

        rows = [
            crop_pool.loc[i]
            for i in seq
        ]

        sequence_name = " > ".join(
            str(r.crop) for r in rows
        )

        fails = []

        # -----------------------------------------------------
        # 3A. DATE-INDEPENDENT ROTATION CHECKS
        # -----------------------------------------------------

        rotation_reasons, health = rotation_check(
            rows,
            cfg,
            site._soil
        )

        fails.extend([
            ("rotation_rules", reason)
            for reason in rotation_reasons
        ])

        names = {
            str(r.crop).strip().lower()
            for r in rows
        }

        for required_crop in prio.must_include:
            if str(required_crop).strip().lower() not in names:
                fails.append((
                    "farmer_priority",
                    f"must-include crop '{required_crop}' missing"
                ))

        food_share = float(
            np.mean([r.is_food for r in rows])
        )

        if (
            food_share < prio.min_food_share
            and prio.min_food_share > 0
        ):
            fails.append((
                "farmer_priority",
                f"food-crop share {food_share:.0%} "
                f"< required {prio.min_food_share:.0%}"
            ))

        if fails:
            rot_reject.append({
                "sequence": sequence_name,
                "checks": ", ".join(sorted({x[0] for x in fails})),
                "reasons": "; ".join(x[1] for x in fails)
            })
            continue

        # -----------------------------------------------------
        # 3B. INITIAL BEAM STATE
        # -----------------------------------------------------

        beam = [{
            "events": [],
            "t": field.start_date,
            "fallow_days": 0,
            "water_sum": 0.0
        }]

        sequence_failures = []
        sequence_explored = 0

        # -----------------------------------------------------
        # 4. EXPAND EACH CROP THROUGH MULTIPLE SOWING DATES
        # -----------------------------------------------------

        for k, crop in enumerate(rows):

            expanded = []

            for state in beam:

                # Land becomes available after previous harvest
                if k == 0:
                    ready = state["t"]
                else:
                    ready = (
                        state["t"]
                        + dt.timedelta(days=cfg.turnaround_days)
                    )

                # Get ALL possible sowing dates in the fallow window
                sowing_options = candidate_sowing_dates(
                    field_ready_date=ready,
                    crop=crop,
                    crop_calender_df=crop_calender_df,
                    terrain_str=field.terrain,
                    max_fallow_days=cfg.max_fallow_days,
                    step_days=date_step_days
                )

                if not sowing_options:
                    sequence_failures.append((
                        "planting_window",
                        f"{crop.crop}: no valid sowing date within "
                        f"{cfg.max_fallow_days} days after {ready}"
                    ))
                    continue

                for sow, calendar_idx in sowing_options:

                    sequence_explored += 1
                    explored_schedules += 1

                    fallow = (sow - ready).days

                    if fallow < 0 or fallow > cfg.max_fallow_days:
                        continue

                    # Get crop duration from this calendar row
                    gp = parse_gp_value(
                        crop_calender_df.loc[calendar_idx, "gp_value"]
                    )

                    if pd.isna(gp) or gp <= 0:
                        sequence_failures.append((
                            "season_duration",
                            f"{crop.crop}: invalid growing period"
                        ))
                        continue

                    harvest = sow + dt.timedelta(days=int(gp))

                    # Rotation horizon check
                    total_days = (
                        harvest - field.start_date
                    ).days

                    if total_days > cfg.max_horizon_days:
                        sequence_failures.append((
                            "season_duration",
                            f"{crop.crop}: rotation exceeds "
                            f"{cfg.max_horizon_days}-day horizon"
                        ))
                        continue

                    # -----------------------------------------
                    # WATER CHECK (CACHED)
                    # -----------------------------------------

                    water_key = (
                        seq[k],
                        sow,
                        harvest,
                        calendar_idx
                    )

                    if water_key not in water_cache:
                        water_cache[water_key] = water_check(
                            crop,
                            sow,
                            harvest,
                            site,
                            field,
                            cfg
                        )

                    water_ok, water_reason, water_metrics = (
                        water_cache[water_key]
                    )

                    if not water_ok:
                        sequence_failures.append((
                            "water",
                            water_reason
                        ))
                        continue

                    # -----------------------------------------
                    # NDVI
                    # -----------------------------------------

                    ndvi_value = site.span_mean(
                        "ndvi",
                        sow,
                        harvest
                    )

                    event = {
                        "crop": crop.crop,
                        "sow": sow,
                        "harvest": harvest,
                        "fallow_d": fallow,
                        "calendar_idx": calendar_idx,
                        **water_metrics,
                        "ndvi": ndvi_value
                    }

                    # -----------------------------------------
                    # CREATE NEW PARTIAL SCHEDULE
                    # -----------------------------------------

                    new_state = {
                        "events": state["events"] + [event],
                        "t": harvest,
                        "fallow_days": (
                            state["fallow_days"] + fallow
                        ),
                        "water_sum": (
                            state["water_sum"]
                            + min(water_metrics["water_ratio"], 1.0)
                        )
                    }

                    expanded.append(new_state)

            # No possible schedule survived this crop stage
            if not expanded:
                beam = []
                break

            # -------------------------------------------------
            # 5. BEAM PRUNING
            # -------------------------------------------------
            # Keep schedules with:
            # - better water adequacy
            # - fewer cumulative fallow days
            #
            # Water is a useful pruning criterion because most
            # other score components are fixed for this sequence.

            def partial_schedule_score(state):
                n_events = len(state["events"])

                average_water = (
                    state["water_sum"] / max(n_events, 1)
                )

                return (
                    average_water
                    - 0.001 * state["fallow_days"]
                )

            expanded.sort(
                key=partial_schedule_score,
                reverse=True
            )

            beam = expanded[:beam_width]

        # -----------------------------------------------------
        # 6. NO COMPLETE SCHEDULE FOUND
        # -----------------------------------------------------

        if not beam:

            if not sequence_failures:
                sequence_failures.append((
                    "scheduling",
                    "No complete schedule survived beam search"
                ))

            unique_failures = list(dict.fromkeys(sequence_failures))

            rot_reject.append({
                "sequence": sequence_name,
                "checks": ", ".join(
                    sorted({x[0] for x in unique_failures})
                ),
                "reasons": (
                    "; ".join(x[1] for x in unique_failures[:8])
                    + f" | explored alternatives: {sequence_explored}"
                )
            })

            continue

        # -----------------------------------------------------
        # 7. SCORE EVERY COMPLETE SCHEDULE IN THE BEAM
        # -----------------------------------------------------

        for state in beam:

            events = state["events"]
            final_harvest = state["t"]

            span = (
                final_harvest - field.start_date
            ).days

            if span > cfg.max_horizon_days:
                continue

            ndvi_values = [
                min(e["ndvi"] / site.ndvi_max, 1.0)
                for e in events
                if (
                    np.isfinite(e["ndvi"])
                    and site.ndvi_max > 0
                )
            ]

            # Component scores
            comp = {
                "soil_fit": float(np.mean([
                    soil_cache[i]["soil_fit"]
                    for i in seq
                ])),

                "water_fit": float(np.mean([
                    min(e["water_ratio"], 1.0)
                    for e in events
                ])),

                "timing": float(
                    sum(
                        (e["harvest"] - e["sow"]).days
                        for e in events
                    ) / max(span, 1)
                ),

                "rotation_health": health,

                "ndvi_support": (
                    float(np.mean(ndvi_values))
                    if ndvi_values else np.nan
                ),

                "profit": float(np.mean([
                    r.income_index for r in rows
                ])),

                "food": food_share
            }

            # Weighted farmer-priority score
            score = sum(
                (
                    0.0 if pd.isna(comp[key])
                    else comp[key]
                ) * weight
                for key, weight in WEIGHTS.items()
            )

            # Warnings
            warnings = []

            if (
                field.irrigation
                and any(
                    e["water_ratio"] < cfg.water_reject_ratio
                    for e in events
                )
            ):
                warnings.append(
                    "needs irrigation in at least one season"
                )

            min_data_coverage = getattr(
                cfg, "min_data_coverage", 0.0
            )

            if site.ndvi_valid_share < min_data_coverage:
                warnings.append("low EO data quality")

            # Store this particular schedule
            rot_accpt.append({
                "sequence": sequence_name,
                "score": round(score, 3),
                **{
                    key: round(value, 2)
                    for key, value in comp.items()
                },
                "total_days": span,
                "fallow_days": state["fallow_days"],
                "warnings": "; ".join(warnings),
                "events": events
            })

    # ---------------------------------------------------------
    # 8. RANK RESULTS
    # ---------------------------------------------------------

    RANKED_COLUMNS = [
        "sequence",
        "score",
        "soil_fit",
        "water_fit",
        "timing",
        "rotation_health",
        "ndvi_support",
        "profit",
        "food",
        "total_days",
        "fallow_days",
        "warnings",
        "events"
    ]

    ranked = pd.DataFrame(
        rot_accpt,
        columns=RANKED_COLUMNS
    )

    if not ranked.empty:

        ranked = ranked.sort_values(
            ["score", "total_days", "fallow_days"],
            ascending=[False, True, True]
        ).reset_index(drop=True)

        ranked.insert(
            0,
            "rank",
            ranked.index + 1
        )

    # ---------------------------------------------------------
    # 9. RETURN RESULTS
    # ---------------------------------------------------------

    rejected_df = pd.DataFrame(
        rot_reject,
        columns=["sequence", "checks", "reasons"]
    )

    feasible_sequences = (
        ranked["sequence"].nunique()
        if not ranked.empty else 0
    )
    
    return {
        "ranked": ranked,
        "top": ranked.head(top_n),
        "rejected": rejected_df,
        "crop_rejections": pd.DataFrame(reject_crop),

        "summary": {
            "candidates": feasible_sequences + len(rot_reject),
            "feasible": feasible_sequences,
            "feasible_schedules": len(rot_accpt),
            "rejected_sequences": len(rot_reject),
            "crops_screened_out": len(reject_crop),
            "crops_in_pool": len(pool_rows),
            "explored_sowing_alternatives": explored_schedules,
            "beam_width": beam_width,
            "date_step_days": date_step_days,
            "weights": {
                key: round(value, 2)
                for key, value in WEIGHTS.items()
            }
        }
    }

In [26]:
from pathlib import Path
import pandas as pd

crop_calender_path = Path().resolve().parent / "data" / "Crop" / "cleaned" / "crop_calender.csv"
crop_calender_df = pd.read_csv(crop_calender_path)
crop_calender_df
"Rice" in crop_calender_df["crop"].values

True

In [27]:
def _norm(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower()

def _yes(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["yes", "y", "true", "1"])
    )

def _int_or_none(x):
    v = pd.to_numeric(x, errors="coerce")
    return None if pd.isna(v) else int(v)

KC_MID = [("green manure", .90), ("pulse", 1.0), ("legume", 1.0), ("oilseed", .95), ("cereal", 1.10),
          ("tuber", 1.10), ("root", 1.05), ("vegetable", 1.05), ("fiber", 1.10), ("fibre", 1.10)]

def prep_crops(crop_dataframe: pd.DataFrame, crop_calender_dataframe: pd.DataFrame, cfg: RotationConfig) -> pd.DataFrame:
    prepared_crops = crop_dataframe.copy().reset_index(drop=True)
    # converting terrain to boolean values
    _terrain_columns = ["terai", "hill", "mountain"]
    
    for terrain_column in _terrain_columns:
        prepared_crops[terrain_column] = _yes(prepared_crops[terrain_column])

    text_columns = ["crop", "category", "family", "function", "species"]
    for column_name in text_columns:
        prepared_crops[column_name] = (
            prepared_crops[column_name]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    prepared_crops["family"] = prepared_crops["family"].str.title()

    fn = prepared_crops["function"].str.lower()
    cat = prepared_crops["category"].str.lower()
    fam = prepared_crops["family"].str.lower()

    prepared_crops["is_green_manure"] = (
        fn.str.contains("green manure", na=False)
    )
    prepared_crops["is_legume"] = (
        fam.eq("fabaceae") |
        fn.str.contains("pulse|legume", na=False)
    )
    prepared_crops["is_cash"] = (
        fn.str.contains("cash|fiber|fibre|oilseed", na=False)
    )
    prepared_crops["is_food"] = (
        fn.str.contains("food|staple|pulse", na=False)
    )
    prepared_crops["heavy_feeder"] = (
        cat.str.contains("cereal|tuber", na=False) &
        ~prepared_crops["is_legume"]
    )

    # Estimate mean crop coefficient if it is not provided
    if "kc_mean" not in prepared_crops.columns:
        def estimate_crop_coefficient(crop_row):
            crop_description = (crop_row["category"] + " " + crop_row["function"])
            matching_coefficient = next(
                (
                    coefficient
                    for keyword, coefficient in KC_MID
                    if keyword in crop_description
                ),
                1.0
            )

            return 0.85 * matching_coefficient

        prepared_crops["kc_mean"] = prepared_crops.apply(
            estimate_crop_coefficient,
            axis=1
        )
    
    # Create income index if it is not provided
    if "income_index" not in prepared_crops.columns:
        prepared_crops["income_index"] = (
            prepared_crops["is_cash"].astype(float)
        )
    
    # Normalize income index
    prepared_crops["income_index"] = (prepared_crops["income_index"] / max(prepared_crops["income_index"].max(), 1e-9))

    prepared_crops["data_flag"] = ""

    needed = {"harv_early_month", "harv_early_day", "sow_early_month", "sow_early_day", "gp_value"}

    if needed.issubset(crop_calender_dataframe.columns):
        cal = crop_calender_dataframe
        cal_crop, cal_aez = _norm(cal["crop"]), _norm(cal["aez"])
        for i, crop_row in prepared_crops.iterrows():
            flags = []
            for terrain in _terrain_columns:
                if not crop_row[terrain]:
                    continue
                rows = cal[(cal_crop == crop_row["crop"].lower()) & (cal_aez == terrain)]
                for _, cr in rows.iterrows():
                    if "all_year" in cal.columns and _norm(pd.Series([cr["all_year"]])).iloc[0] in ("yes", "true", "1", "y"):
                        continue
                    gp = parse_gp_value(cr["gp_value"])
                    parts = [_int_or_none(cr[c]) for c in ("sow_early_month", "sow_early_day",
                                                           "harv_early_month", "harv_early_day")]
                    if gp <= 0 or gp >= 360 or any(p is None for p in parts):
                        continue
                    try:
                        expected = (mk_date(2001, parts[2], parts[3]) - mk_date(2001, parts[0], parts[1])).days % 365
                    except ValueError:
                        continue
                    if abs(expected - min(gp, 364)) > cfg.harvest_mismatch_tol_days:
                        flags.append(f"{terrain}: early harvest implies {expected} d but growing period is {gp} d")
            prepared_crops.loc[i, "data_flag"] = "; ".join(flags)
    return prepared_crops
    

In [28]:
final_crop_k_df = prep_crops(crop_df, crop_calender_df, CFG)
final_crop_k_df.iloc[0]

crop                                                            Rice
species                                              Oryza sativa L.
family                                                       Poaceae
function                                          Cash / Staple Food
terai                                                           True
hill                                                            True
mountain                                                        True
rooting_depth_m                                                  0.5
category                                     Cereals & pseudocereals
ph_opt_min                                                       5.5
ph_opt_max                                                       7.0
ph_abs_min                                                       4.5
ph_abs_max                                                       9.0
agronomic_notes    Highly adaptable to flooded, wet soils across ...
is_green_manure                   

In [29]:
prio = FarmerPriority(profit=0.5, food_security=0.5, soil_health=0.5, water_saving=0.3, low_risk=0.3,
                        must_include=[], exclude=[], min_food_share=0)
fields = FieldProfile(
    field_id="HARATPUR_COORDINATES",
    terrain=get_terrain_elevation(_BHARATPUR_COORDINATES["LAT"], _BHARATPUR_COORDINATES["LON"])["terrain"],
    irrigation=False,
    start_date=dt.date(2026, 3, 1)
)
sensitivity_results = {}
CFG.water_reject_ratio = 0.20

res = assess_rotation(
    final_crop_k_df,
    crop_calender_df,
    site,
    fields,
    prio,
    CFG,
    top_n=20
)
schedule_comparison = []

for _, row in res["top"].iterrows():

    record = {
        "rank": row["rank"],
        "sequence": row["sequence"],
        "score": row["score"],
        "water_fit": row["water_fit"],
        "total_days": row["total_days"],
        "fallow_days": row["fallow_days"]
    }

    for i, event in enumerate(row["events"], start=1):

        record[f"crop_{i}"] = event["crop"]
        record[f"sow_{i}"] = event["sow"]
        record[f"harvest_{i}"] = event["harvest"]

    schedule_comparison.append(record)

schedule_comparison_df = pd.DataFrame(schedule_comparison)

display(schedule_comparison_df)


,rank,sequence,score,water_fit,total_days,fallow_days,crop_1,sow_1,harvest_1,crop_2,sow_2,harvest_2,crop_3,sow_3,harvest_3
0,1,Rice > Mungbean > Mustard,0.794,0.42,303,13,Rice,2026-03-01,2026-06-29,Mungbean,2026-07-09,2026-09-22,Mustard,2026-10-15,2026-12-29
1,2,Corn > Mungbean > Mustard,0.794,0.43,303,23,Corn,2026-03-15,2026-07-03,Mungbean,2026-07-13,2026-09-26,Mustard,2026-10-15,2026-12-29
2,3,Corn > Mungbean > Mustard,0.794,0.43,303,23,Corn,2026-03-08,2026-06-26,Mungbean,2026-07-06,2026-09-19,Mustard,2026-10-15,2026-12-29
3,4,Corn > Mungbean > Mustard,0.794,0.43,303,23,Corn,2026-03-15,2026-07-03,Mungbean,2026-07-15,2026-09-28,Mustard,2026-10-15,2026-12-29
4,5,Rice > Mungbean > Mustard,0.792,0.41,303,13,Rice,2026-03-01,2026-06-29,Mungbean,2026-07-15,2026-09-28,Mustard,2026-10-15,2026-12-29
5,6,Corn > Mungbean > Mustard,0.792,0.41,303,23,Corn,2026-03-08,2026-06-26,Mungbean,2026-07-13,2026-09-26,Mustard,2026-10-15,2026-12-29
6,7,Corn > Mungbean > Mustard,0.792,0.41,303,23,Corn,2026-03-08,2026-06-26,Mungbean,2026-07-15,2026-09-28,Mustard,2026-10-15,2026-12-29
7,8,Corn > Mungbean > Mustard,0.781,0.43,338,58,Corn,2026-03-15,2026-07-03,Mungbean,2026-07-13,2026-09-26,Mustard,2026-11-19,2027-02-02
8,9,Corn > Mungbean > Mustard,0.781,0.43,338,58,Corn,2026-03-08,2026-06-26,Mungbean,2026-07-06,2026-09-19,Mustard,2026-11-19,2027-02-02
9,10,Corn > Mungbean > Mustard,0.781,0.43,338,58,Corn,2026-03-15,2026-07-03,Mungbean,2026-07-15,2026-09-28,Mustard,2026-11-19,2027-02-02


In [30]:
display(res["crop_rejections"])
display(res["rejected"].tail(10))

,crop,checks,reasons
0,Rice,terrain,Rice not recommended for Terai terrain
1,Barley,terrain,Barley not recommended for Terai terrain
2,Barley,terrain,Barley not recommended for Terai terrain
3,Finger Millet,terrain,Finger Millet not recommended for Terai terrain
4,Oat,terrain,Oat not recommended for Terai terrain
5,Soybean,terrain,Soybean not recommended for Terai terrain
6,Pigeonpea,season_duration,No calendar data found for Pigeonpea in Terai
7,Black Gram,terrain,Black Gram not recommended for Terai terrain
8,Dhaincha,season_duration,No calendar data found for Dhaincha in Terai
9,Tobacco,season_duration,No calendar data found for Tobacco in Terai


,sequence,checks,reasons
975,Jute > Jute > Rice,rotation_rules,same species (Corchorus capsularis L.) repeate...
976,Jute > Jute > Wheat,rotation_rules,same species (Corchorus capsularis L.) repeate...
977,Jute > Jute > Corn,rotation_rules,same species (Corchorus capsularis L.) repeate...
978,Jute > Jute > Lentil,rotation_rules,same species (Corchorus capsularis L.) repeate...
979,Jute > Jute > Chickpea,rotation_rules,same species (Corchorus capsularis L.) repeate...
980,Jute > Jute > Mungbean,rotation_rules,same species (Corchorus capsularis L.) repeate...
981,Jute > Jute > Mustard,rotation_rules,same species (Corchorus capsularis L.) repeate...
982,Jute > Jute > Flax,rotation_rules,same species (Corchorus capsularis L.) repeate...
983,Jute > Jute > Potato,rotation_rules,same species (Corchorus capsularis L.) repeate...
984,Jute > Jute > Jute,rotation_rules,same species (Corchorus capsularis L.) repeate...
